# 03｜台灣併購宣告的市場反應：事件研究＋橫斷面回歸（v6，真實資料）

**研究問題**
1. 台灣上市櫃公司宣告併購時，主併公司的股價有沒有異常報酬？
2. 什麼樣的併購會被市場獎勵？（支付方式、是否跨國、是否同業、相對規模）
3. 結果穩不穩？（換窗口、換模型、換樣本、換變數定義）

**輸入**
- `work/events.csv`：上一本 notebook（`build_sample_v6`）產生的事件表
- TEJ 個股日資料（報酬率、成交量），含市場報酬指數 Y9997

**流程（共 12 格，照順序一格一格按）**

| 格 | 做什麼 |
|---|---|
| 1 | 設定研究參數、讀資料 |
| 2 | 對齊交易日、確認每個事件的事件日 |
| 3 | 市場模型：估計每家公司的「正常報酬」 |
| 4 | 算每個事件的累積異常報酬 CAR |
| 5 | 顯著性檢定（主結果） |
| 6 | 每天的平均異常報酬（看反應發生在哪幾天） |
| 7 | 分組比較 |
| 8 | 橫斷面回歸 |
| 9 | 穩健性：換樣本、換變數定義 |
| 10 | 畫圖 |
| 11 | 匯出結果 |
| 12 | 結果摘要（只列事實，結論要自己寫） |

**所有研究設計都在看結果之前就定好了**（寫在第 1 格和第 8、9 格的說明裡）。主結果固定是市場模型的 CAR[-1,+1]；其他版本只用來確認方向一致，不能事後挑顯著的當主結果。

## 第 1 格｜設定研究參數、讀資料

**研究參數**（之後不改）
- 估計期 [-150,-31]：用宣告前約半年的「平常日子」估計這支股票和大盤的關係。停在 -31 是為了不讓宣告前可能走漏消息的那一個月污染估計。
- 估計期至少要有 100 個有效交易日，不夠的事件剔除。
- 主窗口 CAR[-1,+1]；另外三個窗口只做穩健性。

**讀兩份資料**
- `events.csv`：109 宗事件
- 個股日資料：上一本 notebook 已經整理成 `cache/daily.pkl`，直接讀；如果那個檔案不在，就重新讀 8 個年度檔（約 2～3 分鐘）

最後印出大盤報酬的平均和標準差，確認單位是「百分比」（標準差應該在 1 左右）。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf
pd.set_option('display.width', 250); pd.set_option('display.max_rows', 200); pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

BASE = '/content/drive/MyDrive/ma_event_study'
DATA_DIR, WORK_DIR, CACHE_DIR, OUT_DIR = f'{BASE}/data', f'{BASE}/work', f'{BASE}/cache', f'{BASE}/output'
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(CACHE_DIR, exist_ok=True)

# ── 研究設計參數 ──
EST_WINDOW  = (-150, -31)          # 估計期（相對事件日的交易日數）
MIN_EST_OBS = 100                  # 估計期至少要有幾天有效報酬
EVENT_WINDOWS = {'CAR[-1,+1]': (-1, 1), 'CAR[0,+1]': (0, 1), 'CAR[-5,+5]': (-5, 5), 'CAR[-10,+10]': (-10, 10)}
MAIN_WINDOW = 'CAR[-1,+1]'         # 主結果
PLOT_RANGE  = (-10, 10)
MKT_ID = 'Y9997'                   # TEJ 的市場報酬指數（含息）

# ── 事件表 ──
events = pd.read_csv(f'{WORK_DIR}/events.csv', dtype={'event_id': str, 'stock_id': str})
events['t0'] = pd.to_datetime(events['t0']); events['announce_date'] = pd.to_datetime(events['announce_date'])
print(f'事件表：{len(events)} 宗，{events["stock_id"].nunique()} 家公司，事件日 {events["t0"].min().date()} ~ {events["t0"].max().date()}')

# ── 個股日資料 ──
def load_daily():
    cache = f'{CACHE_DIR}/daily.pkl'
    if os.path.exists(cache):
        print('讀取上一本 notebook 整理好的日資料（cache/daily.pkl）')
        return pd.read_pickle(cache)
    files = sorted(f for f in os.listdir(DATA_DIR) if 'daily' in f.lower() and f.lower().endswith('.xlsx'))
    print(f'找不到整理好的檔案，重新讀 {len(files)} 個年度檔（約 2～3 分鐘）')
    parts = []
    for f in files:
        t = time.time()
        df = pd.read_excel(f'{DATA_DIR}/{f}', dtype=str)
        df.columns = [str(c).strip() for c in df.columns]
        ren = {c: v for c in df.columns for k, v in {'代號': 'stock_id', '年月日': 'date', '報酬率': 'ret', '市值': 'mktcap', '成交量': 'vol'}.items() if k in c}
        df = df.rename(columns=ren)
        df['stock_id'] = df['stock_id'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
        for c in ['ret', 'mktcap', 'vol']: df[c] = pd.to_numeric(df[c], errors='coerce')
        parts.append(df[['stock_id', 'date', 'ret', 'mktcap', 'vol']]); print(f'  {f}: {len(df):,} 筆（{time.time() - t:.0f} 秒）')
    daily = pd.concat(parts, ignore_index=True).dropna(subset=['date'])
    daily.to_pickle(cache)
    return daily

daily = load_daily()
print(f'日資料：{len(daily):,} 筆，{daily["stock_id"].nunique():,} 檔證券，{daily["date"].min().date()} ~ {daily["date"].max().date()}')

m = daily.loc[daily['stock_id'] == MKT_ID, 'ret'].dropna()
print(f'\n大盤（{MKT_ID}）日報酬：{len(m):,} 天，平均 {m.mean():.4f}，標準差 {m.std():.4f}，最小 {m.min():.2f}，最大 {m.max():.2f}')
assert len(m) > 1000, f'日資料裡找不到 {MKT_ID} 的報酬，請截圖給 Claude'
assert 0.3 < m.std() < 3, '大盤報酬的標準差不在 0.3～3 之間，單位可能不是百分比，請截圖給 Claude'
print('→ 標準差在 1 左右，單位是百分比（1 代表 1%）。之後所有的 AR、CAR 單位也都是 %')
miss = sorted(set(events['stock_id']) - set(daily['stock_id']))
print('事件公司在日資料裡都找得到' if not miss else f'✗ 這些公司在日資料裡找不到：{miss}')

## 第 2 格｜對齊交易日、確認事件日

- **交易日曆**：大盤有報酬的日子就是交易日。之後講的「-1 日、+1 日」都是交易日，不是日曆日。
- **事件日 t0**：定義是「市場第一個能反應這則消息的交易日」。上一本 notebook 已經處理了盤後和非交易日公告（順延到下一個交易日）。這裡再補一個情況：**盤中公告、但當天整天暫停交易**。那天投資人根本不能買賣，所以第 0 天要再往後移到第一個有成交的交易日（v4 新增，這是我自己在逐格複習時發現的）。
- **成交量為 0 的日子**（那天沒有成交，價格沒有變）分兩種處理：
  - **估計期**：這一天不用，免得把「沒有交易」當成「報酬是 0」，低估這支股票平常的波動。
  - **事件窗口**：這一天的報酬算 0。因為 CAR 是把幾天的報酬加起來，沒成交那天價格沒動，之後恢復交易那天的報酬會把累積的反應一次反映出來，加總起來是對的。

**為什麼事件窗口不能直接丟掉沒成交的日子（v2 的做法，v3 改掉）**：臺灣從 2016 年起，公司董事會決議合併、收購、股份轉換這類重大訊息時要申請**暫停交易**。所以很多併購案在公告當天整天沒有成交。如果窗口內有一天沒成交就把整宗丟掉，丟掉的正好是最正式、最大型的那些併購，樣本會偏掉。

把資料轉成「一列是一天、一欄是一家公司」的表（寬表），之後取某家公司某段期間的報酬比較方便。

In [ ]:
mkt_s = daily[daily['stock_id'] == MKT_ID].dropna(subset=['ret']).drop_duplicates('date').set_index('date')['ret'].sort_index()
calendar = mkt_s.index                                             # 交易日曆
sub = daily[daily['stock_id'].isin(events['stock_id'].unique())].drop_duplicates(['stock_id', 'date']).copy()
sub['traded'] = sub['vol'].fillna(0) > 0
n_zero = int((~sub['traded']).sum())
raw_wide    = sub.pivot(index='date', columns='stock_id', values='ret').reindex(calendar)                                   # TEJ 原始的報酬率
traded_wide = sub.pivot(index='date', columns='stock_id', values='traded').reindex(calendar).fillna(False).astype(bool)     # 那天有沒有成交（沒有資料也算沒成交）
ret_wide     = raw_wide.where(traded_wide)             # 估計期用：沒成交的日子 → 缺值（不用）
ret_evt_wide = raw_wide.where(traded_wide, 0.0)        # 事件窗口用：沒成交的日子 → 報酬 0（價格沒動）
mkt = mkt_s.reindex(calendar)

events['t0_pos'] = calendar.searchsorted(events['t0'])
bad = events[(events['t0_pos'] >= len(calendar)) | (calendar[events['t0_pos'].clip(upper=len(calendar) - 1)] != events['t0'].values)]
assert bad.empty, f'這些事件的 t0 不是交易日：{bad["event_id"].tolist()}，請截圖給 Claude'

# 第 0 天整天沒有成交（盤中公告、但當天暫停交易）→ 市場當天其實沒辦法反應，把第 0 天往後移到第一個有成交的交易日
events['t0_original'] = events['t0']
moved = []
for i, ev in events.iterrows():
    p = p_new = int(ev['t0_pos'])
    while p_new < len(calendar) - 1 and p_new - p < 5 and not traded_wide[ev['stock_id']].iloc[p_new]:
        p_new += 1
    if p_new != p and traded_wide[ev['stock_id']].iloc[p_new]:
        events.loc[i, 't0_pos'] = p_new; events.loc[i, 't0'] = calendar[p_new]
        moved.append((ev['event_id'], ev['firm_name'], ev['announce_timing'], ev['t0'].date(), calendar[p_new].date()))

print(f'交易日曆：{calendar[0].date()} ~ {calendar[-1].date()}，共 {len(calendar):,} 天')
print(f'事件公司的日資料：{len(sub):,} 筆；其中成交量為 0 的有 {n_zero:,} 筆（估計期不用、事件窗口算 0）')
print(f'{len(events)} 宗的 t0 都是交易日：✓')
print(f'第 0 天整天沒有成交、往後移到第一個有成交日的事件：{len(moved)} 宗')
for m in moved: print(f'   編號 {m[0]} {m[1]}（{m[2]}公告）：{m[3]} → {m[4]}')
print('公告時點：', events['announce_timing'].value_counts().to_dict(), '（盤後的 t0 是公告的下一個交易日）')
print(f'最早的事件 t0 = {events["t0"].min().date()}，距離日資料起點 {int(events["t0_pos"].min())} 個交易日（估計期需要 150 個）')

## 第 3 格｜市場模型：估計每家公司的「正常報酬」

對每一個事件，用估計期 [-150,-31] 的資料跑一條迴歸：

> 個股當天報酬 ＝ α ＋ β × 大盤當天報酬 ＋ 誤差

- **β**：大盤漲 1%，這支股票平均跟著漲多少
- **α**：大盤不動時，這支股票平均每天的報酬
- **σ**：誤差的標準差，也就是這支股票「平常有多吵」。後面做標準化檢定時當作單位

有了 α 和 β，事件窗口裡每一天的：
- **正常報酬** ＝ α ＋ β × 當天大盤報酬（如果沒有併購消息，這支股票「應該」有的報酬）
- **異常報酬 AR** ＝ 實際報酬 − 正常報酬（多出來或少掉的部分，歸給併購消息）

同時算另一種比較簡單的 **市場調整模型**：AR ＝ 個股報酬 − 大盤報酬（等於假設 α＝0、β＝1），當穩健性檢驗。

**會被剔除的事件**：估計期有效天數不到 100 天（剛上市、長期停牌或成交太稀疏）。

**最後一段輸出是檢查暫停交易**：列出事件日前後沒有成交的日子。預期會集中在 -1 日（盤後公告的案子，公告當天暫停交易，隔天恢復）。

In [ ]:
MAX_REL = max(max(abs(a), abs(b)) for a, b in list(EVENT_WINDOWS.values()) + [PLOT_RANGE])

panels, params, dropped = [], [], []
for _, ev in events.iterrows():
    sid, p0 = ev['stock_id'], int(ev['t0_pos'])
    if sid not in ret_wide.columns:
        dropped.append((ev['event_id'], ev['firm_name'], '找不到個股報酬資料')); continue
    lo, hi = max(0, p0 + EST_WINDOW[0]), p0 + MAX_REL
    if hi >= len(calendar):
        dropped.append((ev['event_id'], ev['firm_name'], '事件日之後的資料不足')); continue
    seg = pd.DataFrame({'rel_day': np.arange(lo - p0, MAX_REL + 1), 'date': calendar[lo:hi + 1],
                        'ret': ret_wide[sid].values[lo:hi + 1], 'mkt': mkt.values[lo:hi + 1],
                        'ret_evt': ret_evt_wide[sid].values[lo:hi + 1], 'ret_raw': raw_wide[sid].values[lo:hi + 1],
                        'no_trade': ~traded_wide[sid].values[lo:hi + 1]})

    # ── 估計期：算 α、β、σ ──
    est = seg[seg['rel_day'].between(*EST_WINDOW)].dropna(subset=['ret', 'mkt'])
    if len(est) < MIN_EST_OBS:
        dropped.append((ev['event_id'], ev['firm_name'], f'估計期有效天數只有 {len(est)} 天')); continue
    beta, alpha = np.polyfit(est['mkt'], est['ret'], 1)                 # 一條直線：斜率是 β、截距是 α
    resid = est['ret'] - (alpha + beta * est['mkt'])
    T = len(est)
    sigma_mm = np.sqrt((resid ** 2).sum() / (T - 2))                    # 市場模型的殘差標準差
    ar_ma_est = est['ret'] - est['mkt']
    r2 = 1 - (resid ** 2).sum() / ((est['ret'] - est['ret'].mean()) ** 2).sum()

    # ── 事件窗口：算每天的 AR ──
    win = seg[seg['rel_day'] >= -MAX_REL].copy()
    win['AR_MM'] = win['ret_evt'] - (alpha + beta * win['mkt'])         # 市場模型（沒成交的日子實際報酬算 0）
    win['AR_MA'] = win['ret_evt'] - win['mkt']                          # 市場調整模型
    win['event_id'] = ev['event_id']
    panels.append(win)
    params.append(dict(event_id=ev['event_id'], alpha=alpha, beta=beta, r2=r2, n_est=T,
                       sigma_MM=sigma_mm, sigma_MA=ar_ma_est.std(ddof=1),
                       mkt_bar=est['mkt'].mean(), Sxx=((est['mkt'] - est['mkt'].mean()) ** 2).sum(),
                       pos_MM=(resid > 0).mean(), pos_MA=(ar_ma_est > 0).mean()))

ar_panel = pd.concat(panels, ignore_index=True)
params   = pd.DataFrame(params).set_index('event_id')
dropped  = pd.DataFrame(dropped, columns=['event_id', 'firm_name', 'reason'])

print(f'有效事件：{len(params)} 宗；剔除：{len(dropped)} 宗')
if len(dropped): display(dropped)
print('\n【市場模型的估計結果】（每一宗各估一組，這裡看整體的分布）')
display(params[['alpha', 'beta', 'r2', 'sigma_MM', 'n_est']].describe().loc[['mean', 'min', '25%', '50%', '75%', 'max']])
print(f'估計期不到 120 天的事件：{int((params["n_est"] < 120).sum())} 宗（最少 {int(params["n_est"].min())} 天）')

# ── 事件窗口裡沒有成交的日子 ──
nt = ar_panel[ar_panel['no_trade']].merge(events[['event_id', 'firm_name', 'payment', 'announce_timing']], on='event_id')
print('\n【事件前後 5 天內，沒有成交的日子】各相對日有幾宗（-1 日是盤後公告的公告當天）：')
print('  ', nt[nt['rel_day'].between(-5, 5)].groupby('rel_day').size().reindex(range(-5, 6), fill_value=0).to_dict())
nt3 = nt[nt['rel_day'].between(-1, 1)]
no_trade_3d = ar_panel[ar_panel['rel_day'].between(-1, 1)].groupby('event_id')['no_trade'].any().reindex(params.index)
print(f'[-1,+1] 內至少有一天沒成交的事件：{int(no_trade_3d.sum())} 宗；支付方式：', events.set_index('event_id').loc[no_trade_3d[no_trade_3d].index, 'payment'].value_counts().to_dict())
print('這些日子 TEJ 原始報酬率的分布（應該是 0 或缺值）：', nt3['ret_raw'].round(2).value_counts(dropna=False).head(6).to_dict())
display(nt3.groupby('event_id').agg(firm=('firm_name', 'first'), payment=('payment', 'first'), timing=('announce_timing', 'first'),
                                    no_trade_days=('rel_day', lambda s: ', '.join(f'{int(x):+d}' for x in s)), first_date=('date', 'min')).sort_values('first_date'))

## 第 4 格｜累積異常報酬 CAR

**CAR ＝ 把窗口內每一天的 AR 加起來。** CAR[-1,+1] 就是事件日前一天、當天、後一天三天的 AR 總和。

- 每個事件、每個窗口各算一個 CAR，單位是 %。
- 窗口內沒有成交的日子報酬算 0（見第 2 格的說明），所以暫停交易的案子也算得出 CAR。只有真的缺資料時，那個窗口才不算。

**標準化 CAR（SCAR）**：把 CAR 除以「這支股票在這幾天本來就會有的波動」，給第 5 格的 BMP 檢定用。
分母不只是 σ×√天數，還加上「預測誤差修正」：α、β 是估計出來的，本身有誤差，窗口越長、估計期越短，這個誤差越大。

最後列出 CAR 最高和最低的各 5 宗。**這一步是在檢查資料**：極端值有可能是真的大漲大跌，也有可能是資料或事件日有問題，要逐宗看過。

In [ ]:
def calc_car(panel, ar_col):
    out = {}
    for name, (a, b) in EVENT_WINDOWS.items():
        w = panel[panel['rel_day'].between(a, b)]
        out[name] = w.groupby('event_id')[ar_col].sum(min_count=b - a + 1)      # 少一天就不算
    return pd.DataFrame(out).reindex(params.index)

car_mm, car_ma = calc_car(ar_panel, 'AR_MM'), calc_car(ar_panel, 'AR_MA')

# 標準化 CAR：CAR ÷ 它的標準差
scar_mm, scar_ma = pd.DataFrame(index=params.index), pd.DataFrame(index=params.index)
for name, (a, b) in EVENT_WINDOWS.items():
    L = b - a + 1
    w = ar_panel[ar_panel['rel_day'].between(a, b)]
    mkt_dev = (w.groupby('event_id')['mkt'].sum(min_count=L).reindex(params.index) - L * params['mkt_bar'])   # 窗口內大盤報酬偏離估計期平均的總和
    var_mm = params['sigma_MM'] ** 2 * (L + L ** 2 / params['n_est'] + mkt_dev ** 2 / params['Sxx'])        # 含預測誤差修正
    scar_mm[name] = car_mm[name] / np.sqrt(var_mm)
    scar_ma[name] = car_ma[name] / (params['sigma_MA'] * np.sqrt(L))

print('【CAR 的分布】市場模型，單位 %')
display(car_mm.describe(percentiles=[.05, .25, .5, .75, .95]).loc[['count', 'mean', 'std', 'min', '5%', '25%', '50%', '75%', '95%', 'max']])

info = events.set_index('event_id')[['firm_name', 'target_name', 't0', 'payment', 'rel_size']]
look = info.join(car_mm[[MAIN_WINDOW, 'CAR[-5,+5]']], how='inner').join(params[['beta', 'n_est']]).dropna(subset=[MAIN_WINDOW]).sort_values(MAIN_WINDOW)
print(f'【{MAIN_WINDOW} 最低的 5 宗】'); display(look.head(5))
print(f'【{MAIN_WINDOW} 最高的 5 宗】'); display(look.tail(5).iloc[::-1])
print('各窗口算不出 CAR 的事件數（窗口內真的缺資料）：', car_mm.isna().sum().to_dict())

## 第 5 格｜顯著性檢定（主結果）

平均 CAR 不等於 0，是真的有反應，還是剛好抽到的？用四種檢定互相對照：

| 檢定 | 在問什麼 | 特性 |
|---|---|---|
| **t 檢定** | 平均 CAR 是不是 0 | 最基本；容易被極端值和高波動的股票影響 |
| **BMP 標準化檢定** | 以各家公司平常的波動為單位，平均反應是不是 0 | 高波動的股票不會主導結果；宣告期間波動變大時仍然可靠 |
| **廣義符號檢定** | CAR 為正的宗數，比「平常日子」預期的多還是少 | 只看正負，不受極端值影響 |
| **Wilcoxon 符號等級檢定** | 中位數是不是 0 | 只用排名，不受極端值影響 |

**和筆記不同的兩個地方（v2 改得比較嚴謹）**
- BMP 的標準化加上了預測誤差修正（第 4 格）。
- 符號檢定的基準不是 50%，而是每家公司在估計期「AR 為正的天數比例」的平均。股票報酬通常右偏，平常日子 AR 為正的比例會略低於 50%，用 50% 當基準會偏保守。

**怎麼讀**：p < 0.05 顯著；0.05～0.10 邊際顯著；> 0.10 不顯著。四種結論一致，結果才可信。
同時報告 **95% 信賴區間**：資料和「真實的平均效果落在這個範圍內」都不衝突。區間比「顯著／不顯著」有用。

In [ ]:
def test_car(car, scar, pos_frac):
    x = car.dropna(); n = len(x)
    z = scar.loc[x.index]
    se = x.std(ddof=1) / np.sqrt(n)
    tcrit = stats.t.ppf(0.975, n - 1)
    t_bmp = z.mean() / (z.std(ddof=1) / np.sqrt(n))
    n_pos, p_hat = int((x > 0).sum()), pos_frac.loc[x.index].mean()                 # 平常日子 AR 為正的比例
    z_sign = (n_pos - n * p_hat) / np.sqrt(n * p_hat * (1 - p_hat))
    return {'N': n, 'Mean': x.mean(), 'CI low': x.mean() - tcrit * se, 'CI high': x.mean() + tcrit * se, 'Median': x.median(),
            '% pos': n_pos / n * 100, 't': x.mean() / se, 'p (t)': 2 * stats.t.sf(abs(x.mean() / se), n - 1),
            't (BMP)': t_bmp, 'p (BMP)': 2 * stats.t.sf(abs(t_bmp), n - 1),
            'p (sign)': 2 * stats.norm.sf(abs(z_sign)), 'p (Wilcoxon)': stats.wilcoxon(x).pvalue}

MODELS = {'Market model': (car_mm, scar_mm, params['pos_MM']), 'Market-adjusted': (car_ma, scar_ma, params['pos_MA'])}
rows = [{'Model': mname, 'Window': w, **test_car(car[w], scar[w], pos)} for mname, (car, scar, pos) in MODELS.items() for w in EVENT_WINDOWS]
robust = pd.DataFrame(rows).set_index(['Model', 'Window'])

main = robust.loc[('Market model', MAIN_WINDOW)]
print(f'▶ 主結果：{MAIN_WINDOW}，市場模型，N = {int(main["N"])}')
print(f'   平均 CAR = {main["Mean"]:+.2f}%，95% 信賴區間 [{main["CI low"]:+.2f}%, {main["CI high"]:+.2f}%]，中位數 {main["Median"]:+.2f}%，{main["% pos"]:.1f}% 的事件為正')
print(f'   t = {main["t"]:.2f}（p = {main["p (t)"]:.3f}）｜BMP t = {main["t (BMP)"]:.2f}（p = {main["p (BMP)"]:.3f}）｜符號檢定 p = {main["p (sign)"]:.3f}｜Wilcoxon p = {main["p (Wilcoxon)"]:.3f}')
print(f'   符號檢定的基準：平常日子 AR 為正的比例平均是 {params["pos_MM"].mean() * 100:.1f}%')
print('\n▶ 所有窗口 × 兩種模型（Mean、CI、Median 的單位是 %）')
display(robust)

## 第 6 格｜每天的平均異常報酬

- **AAR**：某一天（例如 +1 日）所有事件 AR 的平均。看市場的反應集中在哪一天。
- **CAAR**：從 -10 日開始把 AAR 一路加起來。第 10 格會畫成圖。

**看什麼**
- 反應是不是集中在 0、+1 日（盤後公告的反應在 t0 當天，因為 t0 已經順延過了）
- -10 到 -2 日有沒有明顯的走勢：有的話可能是消息提前走漏
- +2 日之後有沒有反轉

In [ ]:
rng = ar_panel[ar_panel['rel_day'].between(*PLOT_RANGE)]
daily_ar = rng.groupby('rel_day')['AR_MM'].agg(AAR='mean', sd='std', N='count')
daily_ar['t'] = daily_ar['AAR'] / (daily_ar['sd'] / np.sqrt(daily_ar['N']))
daily_ar['p'] = 2 * stats.t.sf(daily_ar['t'].abs(), daily_ar['N'] - 1)
daily_ar['% pos'] = rng.groupby('rel_day')['AR_MM'].apply(lambda s: (s > 0).mean() * 100)

def caar_path(ar_col):
    w = rng.pivot(index='rel_day', columns='event_id', values=ar_col)
    w = w.loc[:, w.notna().all()].cumsum()                          # 只用整段期間都有資料的事件，每一天的樣本才一樣
    mean, se = w.mean(axis=1), w.std(axis=1, ddof=1) / np.sqrt(w.shape[1])
    return pd.DataFrame({'CAAR': mean, 'lo': mean - 1.96 * se, 'hi': mean + 1.96 * se}), w.shape[1]

(caar_mm, n_path), (caar_ma, _) = caar_path('AR_MM'), caar_path('AR_MA')
daily_ar['CAAR'] = caar_mm['CAAR']
print(f'【每天的平均異常報酬】市場模型，單位 %（CAAR 用整段都有資料的 {n_path} 宗計算）')
display(daily_ar[['N', 'AAR', 't', 'p', '% pos', 'CAAR']])

## 第 7 格｜分組比較

把事件照一個特徵分成兩組，比較兩組的平均 CAR[-1,+1]。這是最直觀的看法，也最容易講給沒有財金背景的人聽。

| 特徵 | 「是」的定義 |
|---|---|
| 現金支付 | 全部用現金（發行新股、現金加股票都算「否」。混合支付只有 4 宗，不夠獨立成一組，而且只要有發股票就有「股價是否偏高」的訊號） |
| 跨國 | 標的主要營運地在臺灣以外 |
| 同業 | 標的和主併公司做同一類產品或服務（上下游、無關都算「否」） |
| 大型交易 | 相對規模高於樣本中位數 |
| 標的為上市櫃公司 | 標的有公開股價 |

兩種檢定：Welch t 檢定（比平均）、Mann-Whitney 檢定（比排名，不受極端值影響）。

**限制**：分組比較沒有控制其他特徵。例如現金交易的規模通常比較小，現金組和股票組的差距裡混了規模的影響。所以下一格要用迴歸。

In [ ]:
cs = events.set_index('event_id').loc[params.index].copy()
cs['cash']          = (cs['payment'] == 'cash').astype(int)
cs['same_industry'] = (cs['relatedness'] == 'same_industry').astype(int)
cs['related']       = (cs['relatedness'] != 'unrelated').astype(int)          # 另一種定義：同業或上下游
cs['log_rel_size']  = np.log(cs['rel_size'])
cs['log_mktcap']    = np.log(cs['acq_mktcap_mn'])
cs['flag_any']      = cs[[c for c in cs.columns if c.startswith('flag_')]].max(axis=1)
cs['covid']         = cs['t0'].between('2020-01-20', '2020-04-30').astype(int)
cs['no_trade_3d']   = no_trade_3d.astype(int)                                   # [-1,+1] 內有沒有成交的日子（多半是暫停交易）
y_main = car_mm[MAIN_WINDOW]

GROUPS = {'Cash payment': cs['cash'] == 1, 'Cross-border': cs['cross_border'] == 1, 'Same industry': cs['same_industry'] == 1,
          'Large deal (above-median relative size)': cs['rel_size'] > cs['rel_size'].median(), 'Listed target': cs['target_listed'] == 1}
sub_rows = []
for label, mask in GROUPS.items():
    g1, g0 = y_main[mask].dropna(), y_main[~mask].dropna()
    sub_rows.append({'Characteristic': label, 'N yes': len(g1), 'Mean yes': g1.mean(), 'Median yes': g1.median(),
                     'N no': len(g0), 'Mean no': g0.mean(), 'Median no': g0.median(), 'Diff': g1.mean() - g0.mean(),
                     'p (Welch t)': stats.ttest_ind(g1, g0, equal_var=False).pvalue, 'p (Mann-Whitney)': stats.mannwhitneyu(g1, g0).pvalue,
                     'se yes': g1.std(ddof=1) / np.sqrt(len(g1)), 'se no': g0.std(ddof=1) / np.sqrt(len(g0))})
subgroup = pd.DataFrame(sub_rows).set_index('Characteristic')
print(f'【分組比較】{MAIN_WINDOW}，市場模型，單位 %')
display(subgroup.drop(columns=['se yes', 'se no']))

X_MAIN = ['cash', 'cross_border', 'same_industry', 'log_rel_size', 'log_mktcap']
print('【解釋變數之間的相關係數】（太高的話迴歸會分不清楚是誰的影響；絕對值超過 0.6 要留意）')
display(cs[X_MAIN + ['target_listed', 'related_party']].corr().round(2))

## 第 8 格｜橫斷面回歸：什麼樣的併購被市場獎勵

> CAR ＝ b0 ＋ b1·現金 ＋ b2·跨國 ＋ b3·同業 ＋ b4·ln(相對規模) ＋ b5·ln(市值) ＋ 誤差

迴歸把其他特徵「固定住」，單獨看每一個特徵的影響。例如 b1 是「跨不跨國、是不是同業、規模、公司大小都一樣的兩宗併購，現金的那一宗 CAR 平均高多少個百分點」。

| 變數 | 定義 | 文獻上的預期 |
|---|---|---|
| `cash` | 全現金 ＝ 1 | 正：用股票付款可能被解讀成管理層認為自家股價偏高 |
| `cross_border` | 跨國 ＝ 1 | 不確定 |
| `same_industry` | 同業 ＝ 1 | 正：綜效比較容易實現 |
| `log_rel_size` | ln(交易金額 ÷ 主併公司市值) | 不確定：交易越大，好壞都被放大 |
| `log_mktcap` | ln(主併公司市值)，控制變數 | 負：大公司的併購報酬通常較低 |

（文獻的出處寫進報告前要找原文核對。）

- **標準誤用 HC1**：小公司的 CAR 波動大、大公司小，誤差的大小不一致。一般的標準誤在這種情況會失準，HC1 仍然可靠。
- **取對數的變數怎麼讀**：係數 × 0.693 ＝「這個變數變成 2 倍時，CAR 變動幾個百分點」。

**四個版本並排**：右邊的解釋變數都一樣，只換左邊 CAR 的算法，一次只改一項。

| 版本 | 改了什麼 | 回應的質疑 |
|---|---|---|
| (1) | 主規格：市場模型、CAR[-1,+1] | — |
| (2) | 窗口改成 [-5,+5] | 結果是不是只在三天窗口才成立？ |
| (3) | 改用市場調整模型 | 是不是 β 估不準造成的？ |
| (4) | CAR 縮尾（最高、最低各 5% 拉回界線） | 是不是被少數極端事件撐起來的？ |

In [ ]:
def run_reg(y, xvars=X_MAIN, data=None, extra=''):
    d = (cs if data is None else data).assign(y=y).dropna(subset=['y'] + xvars)
    return smf.ols('y ~ ' + ' + '.join(xvars) + extra, data=d).fit(cov_type='HC1', use_t=True)   # HC1 穩健標準誤；p 值用 t 分配（小樣本比較保守）

def winsorize(s, q=0.05):
    return s.clip(s.quantile(q), s.quantile(1 - q))

def stars(p):
    return '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else ''

def reg_table(models, names):
    tab = {}
    for k, m in models.items():
        col = {}
        for v in ['Intercept'] + names:
            if v in m.params.index:
                col[v] = f'{m.params[v]:.3f}{stars(m.pvalues[v])}'; col[v + ' (t)'] = f'({m.tvalues[v]:.2f})'
            else:
                col[v] = ''; col[v + ' (t)'] = ''
        col['N'] = f'{int(m.nobs)}'; col['R²'] = f'{m.rsquared:.3f}'; col['Adj. R²'] = f'{m.rsquared_adj:.3f}'
        tab[k] = col
    return pd.DataFrame(tab)

models_a = {'(1) Main': run_reg(car_mm[MAIN_WINDOW]), '(2) CAR[-5,+5]': run_reg(car_mm['CAR[-5,+5]']),
            '(3) Market-adjusted': run_reg(car_ma[MAIN_WINDOW]), '(4) Winsorized': run_reg(winsorize(car_mm[MAIN_WINDOW]))}
reg_a = reg_table(models_a, X_MAIN)
print('【迴歸表 A：換 CAR 的算法】係數單位是 CAR 的百分點；括號內是 HC1 穩健 t 值；*** p<0.01、** p<0.05、* p<0.1')
display(reg_a)

m1 = models_a['(1) Main']
ci = m1.conf_int()
print('【主規格 (1) 的係數、95% 信賴區間、p 值】')
display(pd.DataFrame({'coef': m1.params, 'CI low': ci[0], 'CI high': ci[1], 't': m1.tvalues, 'p': m1.pvalues}))
print(f'整條迴歸的 F 檢定 p = {float(m1.f_pvalue):.3f}（五個變數合起來有沒有解釋力）')

## 第 9 格｜穩健性：換樣本、換變數定義

第 8 格換的是「CAR 怎麼算」。這一格換的是「用哪些事件」和「變數怎麼定義」。這些版本都是審閱樣本時就決定要做的，不是看到結果才加的。

**表 B：平均 CAR 在不同樣本下的結果**

| 樣本 | 為什麼要看 |
|---|---|
| 排除關係人交易 | 向董事、大股東或關聯企業買的，價格和動機可能和一般併購不同 |
| 排除 KY 公司 | 註冊在海外、營運多在大陸或東南亞，「跨國」的意義不一樣 |
| 排除邊界個案 | 審閱時標記過的（產業、持股、跨國認定有疑義；規模只算到下限；公告前已有媒體報導） |
| 排除 2020/1/20～4/30 | 疫情初期市場劇烈波動，市場模型在這段期間可能失準 |
| 排除窗口內有沒成交日子的 | 暫停交易或成交清淡的案子，沒成交那天的報酬是用 0 補的；拿掉它們看結果會不會變 |
| 只看現金／只看有發股票 | 兩種支付方式各自的反應 |

**表 C：迴歸在不同設定下的結果**

| 版本 | 改了什麼 |
|---|---|
| (5) | 產業關係改用另一種定義：同業或上下游 ＝ 1（只有「無關」是 0） |
| (6) | 多放兩個控制變數：標的是否上市櫃、是否關係人交易 |
| (7) | 排除關係人交易 |
| (8) | 排除 KY 公司 |
| (9) | 排除邊界個案 |
| (10) | 加入年度固定效果（每一年各有自己的基準，排除「某一年整體行情特別好或特別差」的影響） |

**怎麼讀**：看每個變數的**方向**在各版本是不是一致、大小有沒有大幅改變。版本多，偶然出現一兩個星號是正常的，不能挑出來講。

In [ ]:
SAMPLES = {'Full sample': pd.Series(True, index=cs.index), 'Excl. related-party deals': cs['related_party'] == 0, 'Excl. KY firms': cs['ky_firm'] == 0,
           'Excl. flagged borderline cases': cs['flag_any'] == 0, 'Excl. COVID crash (2020/1/20-4/30)': cs['covid'] == 0,
           'Excl. deals with a no-trade day in [-1,+1]': cs['no_trade_3d'] == 0,
           'Cash deals only': cs['cash'] == 1, 'Stock or mixed deals only': cs['cash'] == 0}
rob_sample = pd.DataFrame([{'Sample': k, **test_car(car_mm.loc[mask, MAIN_WINDOW], scar_mm.loc[mask, MAIN_WINDOW], params.loc[mask, 'pos_MM'])}
                           for k, mask in SAMPLES.items()]).set_index('Sample')
print(f'【表 B：平均 {MAIN_WINDOW} 在不同樣本下的結果】市場模型，單位 %')
display(rob_sample)

X_ALT = ['cash', 'cross_border', 'related', 'log_rel_size', 'log_mktcap']
X_EXT = X_MAIN + ['target_listed', 'related_party']
y = car_mm[MAIN_WINDOW]
models_b = {'(1) Main': m1, '(5) Alt. industry def.': run_reg(y, X_ALT), '(6) More controls': run_reg(y, X_EXT),
            '(7) Excl. related-party': run_reg(y, data=cs[cs['related_party'] == 0]), '(8) Excl. KY': run_reg(y, data=cs[cs['ky_firm'] == 0]),
            '(9) Excl. flagged': run_reg(y, data=cs[cs['flag_any'] == 0]), '(10) Year FE': run_reg(y, extra=' + C(year)')}
reg_b = reg_table(models_b, ['cash', 'cross_border', 'same_industry', 'related', 'log_rel_size', 'log_mktcap', 'target_listed', 'related_party'])
print('【表 C：迴歸在不同設定下的結果】係數單位是 CAR 的百分點；括號內是 HC1 穩健 t 值；(10) 的年度虛擬變數不列出')
display(reg_b)

# 每個變數在 10 個版本裡的方向和顯著次數
all_models = {**models_a, **{k: v for k, v in models_b.items() if k != '(1) Main'}}
summ = []
for v in X_MAIN:
    ms = [m for m in all_models.values() if v in m.params.index]
    summ.append({'變數': v, '出現在幾個版本': len(ms), '係數為正': sum(m.params[v] > 0 for m in ms), '係數為負': sum(m.params[v] < 0 for m in ms),
                 'p<0.10 的版本數': sum(m.pvalues[v] < 0.10 for m in ms), 'p<0.05 的版本數': sum(m.pvalues[v] < 0.05 for m in ms),
                 '係數最小': min(m.params[v] for m in ms), '係數最大': max(m.params[v] for m in ms)})
sign_summary = pd.DataFrame(summ).set_index('變數')
print('【每個變數在各版本的方向與顯著性】'); display(sign_summary)

## 第 10 格｜畫圖

三張圖，標籤用英文（要放 GitHub 和英文摘要）：

1. **CAAR 走勢圖**：-10 到 +10 日的累積平均異常報酬，陰影是 95% 信賴區間。兩種模型並列。
2. **分組比較圖**：五個特徵「是／否」兩組的平均 CAR[-1,+1]，橫線是 95% 信賴區間。
3. **CAR 分布圖**：每一宗的 CAR[-1,+1] 落在哪裡，看結果是不是被少數極端值主導。

圖會存到 Drive 的 `ma_event_study/output/`。

In [ ]:
BLUE, ORANGE = '#2a78d6', '#eb6834'
INK, INK2, MUTED, GRID, SURF = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#fcfcfb'
plt.rcParams.update({'figure.facecolor': SURF, 'axes.facecolor': SURF, 'savefig.facecolor': SURF, 'axes.edgecolor': '#c3c2b7', 'axes.labelcolor': INK2,
                     'xtick.color': MUTED, 'ytick.color': MUTED, 'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.6,
                     'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 10, 'axes.titleweight': 'bold',
                     'axes.titlecolor': INK, 'axes.titlelocation': 'left', 'axes.titlepad': 28, 'axes.axisbelow': True})
def subtitle(ax, text):
    ax.text(0, 1.03, text, transform=ax.transAxes, color=INK2, fontsize=9.5, va='bottom')

# ── 圖 1：CAAR 走勢 ──
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.fill_between(caar_mm.index, caar_mm['lo'], caar_mm['hi'], color=BLUE, alpha=0.15, linewidth=0, label='95% CI (market model)')
ax.plot(caar_mm.index, caar_mm['CAAR'], color=BLUE, lw=2, marker='o', ms=4, label='Market model')
ax.plot(caar_ma.index, caar_ma['CAAR'], color=ORANGE, lw=2, ls='--', label='Market-adjusted')
ax.axhline(0, color='#c3c2b7', lw=1); ax.axvline(0, color=MUTED, lw=1, ls=':')
ax.set_xlabel('Trading days relative to announcement (day 0 = first trading day the market could react)'); ax.set_ylabel('CAAR (%)')
ax.set_title('Cumulative average abnormal return around M&A announcements')
subtitle(ax, f'Taiwan-listed acquirers, 2019-2023H1, N = {n_path} deals with complete data over [-10,+10]')
ax.set_xticks(range(PLOT_RANGE[0], PLOT_RANGE[1] + 1, 2))
ax.legend(frameon=False, labelcolor=INK2, loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=3)        # 圖例放在圖的下方，不會壓到線
fig.tight_layout(); fig.savefig(f'{OUT_DIR}/fig1_caar.png', dpi=200, bbox_inches='tight'); plt.show()

# ── 圖 2：分組比較 ──
short = ['Cash payment', 'Cross-border', 'Same industry', 'Large deal', 'Listed target']
labels = [f'{s}\n(diff. p = {p:.2f})' for s, p in zip(short, subgroup['p (Welch t)'])]
yy = np.arange(len(subgroup))
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(yy - 0.19, subgroup['Mean yes'], height=0.34, color=BLUE, label='Yes', xerr=1.96 * subgroup['se yes'], error_kw={'ecolor': INK2, 'lw': 1, 'capsize': 3})
ax.barh(yy + 0.19, subgroup['Mean no'], height=0.34, color=ORANGE, label='No', xerr=1.96 * subgroup['se no'], error_kw={'ecolor': INK2, 'lw': 1, 'capsize': 3})
ax.set_yticks(yy, labels, color=INK2); ax.invert_yaxis()
ax.axvline(0, color='#c3c2b7', lw=1); ax.grid(axis='y', visible=False)
ax.set_xlabel(f'Mean {MAIN_WINDOW} (%), market model; error bars show 95% CI')
ax.set_title('Announcement returns by deal characteristic')
subtitle(ax, f'N = {int(y_main.notna().sum())} deals; p-values from Welch t-tests of the difference in means')
ax.legend(frameon=False, labelcolor=INK2, loc='lower right', bbox_to_anchor=(1.0, 1.0), ncol=2, borderaxespad=0.2)   # 圖例放在圖外，不會壓到誤差線
fig.tight_layout(); fig.savefig(f'{OUT_DIR}/fig2_subgroups.png', dpi=200, bbox_inches='tight'); plt.show()

# ── 圖 3：CAR 分布 ──
x = y_main.dropna()
fig, ax = plt.subplots(figsize=(8, 3.9))
ax.hist(x, bins=24, color=BLUE, edgecolor=SURF, linewidth=1.5)
ax.axvline(0, color='#c3c2b7', lw=1)
ax.axvline(x.mean(), color=INK, lw=1.5, ls='--'); ax.axvline(x.median(), color=INK2, lw=1.2, ls=':')
top = ax.get_ylim()[1]; ax.set_ylim(0, top * 1.22)                                      # 上方留白放文字，不會壓到長條
mean_right = x.mean() >= x.median()                                                     # 平均和中位數的標籤一左一右，不會疊在一起
LABEL_BOX = dict(boxstyle='square,pad=0.15', facecolor=SURF, edgecolor='none')            # 標籤加上和背景同色的底，0 的那條線、格線不會穿過文字（v6）
ax.annotate(f'mean = {x.mean():+.2f}%', (x.mean(), top * 1.12), xytext=(6 if mean_right else -6, 0), textcoords='offset points',
            color=INK, ha='left' if mean_right else 'right', va='center', bbox=LABEL_BOX)        # 標籤和虛線之間留 6 點的空隙；底色蓋住後面的格線
ax.annotate(f'median = {x.median():+.2f}%', (x.median(), top * 1.12), xytext=(-6 if mean_right else 6, 0), textcoords='offset points',
            color=INK2, ha='right' if mean_right else 'left', va='center', bbox=LABEL_BOX)
ax.set_xlabel(f'{MAIN_WINDOW} (%), market model'); ax.set_ylabel('Number of deals'); ax.grid(axis='x', visible=False)
ax.set_title(f'Distribution of {MAIN_WINDOW} across deals')
subtitle(ax, f'N = {len(x)} deals; {int((x > 0).sum())} positive, {int((x < 0).sum())} negative')
fig.tight_layout(); fig.savefig(f'{OUT_DIR}/fig3_car_distribution.png', dpi=200, bbox_inches='tight'); plt.show()
print('已存檔：', sorted(f for f in os.listdir(OUT_DIR) if f.endswith('.png')))

## 第 11 格｜匯出結果

存成兩個 Excel 檔到 Drive 的 `ma_event_study/output/`：

- `results_full.xlsx`：包含每一宗事件的 α、β、各窗口的 CAR。**只放 Drive**（由 TEJ 的報酬率算出來的逐筆資料）。
- `results_tables.xlsx`：只有彙總的表（檢定、每日 AAR、分組比較、三張迴歸表、樣本篩選過程）。這一份可以放 GitHub。

In [ ]:
per_event = (cs[['stock_id', 'firm_name', 'announce_date', 't0', 'target_name', 'payment', 'cross_border', 'relatedness', 'target_listed', 'rel_size', 'acq_mktcap_mn',
                 'related_party', 'ky_firm', 'flag_any', 'no_trade_3d']]
             .join(params[['alpha', 'beta', 'r2', 'sigma_MM', 'n_est']]).join(car_mm.add_prefix('MM_')).join(car_ma.add_prefix('MA_')))
funnel = pd.read_csv(f'{WORK_DIR}/sample_funnel.csv')
if len(dropped):
    funnel.loc[len(funnel)] = ('  排除：估計期有效天數不足', -len(dropped), ''); funnel.loc[len(funnel)] = ('進入事件研究的樣本', len(params), '')

TABLES = {'sample_funnel': funnel, 'tests_by_window': robust.reset_index(), 'daily_AAR': daily_ar.reset_index(), 'subgroups': subgroup.drop(columns=['se yes', 'se no']).reset_index(),
          'tests_by_sample': rob_sample.reset_index(), 'reg_A_car_definition': reg_a.reset_index(), 'reg_B_sample_and_vars': reg_b.reset_index(),
          'reg_sign_summary': sign_summary.reset_index()}
with pd.ExcelWriter(f'{OUT_DIR}/results_tables.xlsx') as xw:
    for name, df in TABLES.items(): df.to_excel(xw, sheet_name=name, index=False)
with pd.ExcelWriter(f'{OUT_DIR}/results_full.xlsx') as xw:
    per_event.to_excel(xw, sheet_name='per_event_CAR')
    dropped.to_excel(xw, sheet_name='dropped', index=False)
    for name, df in TABLES.items(): df.to_excel(xw, sheet_name=name, index=False)
print('已存檔：', sorted(os.listdir(OUT_DIR)))

## 第 12 格｜結果摘要

把前面各表的重點數字整理成幾行文字，存成 `output/summary_facts.txt`。

**這一格只列事實，不下結論。** 結論要等你和 Claude 一起看過每一張表之後再寫，原則是：
- 顯著（p < 0.05）而且各版本方向一致、有文獻或道理支持的，才寫成發現。
- 不顯著的寫「沒有足夠證據」，並附上信賴區間；不能寫成「沒有影響」。
- 事件研究量到的是**宣告當下市場的預期**，不是併購之後的實際績效。

In [ ]:
NICE = {'cash': '現金支付', 'cross_border': '跨國', 'same_industry': '同業', 'log_rel_size': '相對規模', 'log_mktcap': '主併公司市值'}
def sig_word(p): return '顯著' if p < 0.05 else '邊際顯著' if p < 0.10 else '不顯著'

L = [f'【樣本】{len(events)} 宗事件進入計算，{len(dropped)} 宗因估計期資料不足剔除，有效 {len(params)} 宗（{cs["stock_id"].nunique()} 家公司），事件日 {cs["t0"].min().date()} ~ {cs["t0"].max().date()}。', '',
     f'【主結果】{MAIN_WINDOW}（市場模型，N = {int(main["N"])}）：平均 {main["Mean"]:+.2f}%，95% 信賴區間 [{main["CI low"]:+.2f}%, {main["CI high"]:+.2f}%]，中位數 {main["Median"]:+.2f}%，{main["% pos"]:.1f}% 為正。',
     f'  t 檢定 p = {main["p (t)"]:.3f}（{sig_word(main["p (t)"])}）；BMP p = {main["p (BMP)"]:.3f}（{sig_word(main["p (BMP)"])}）；符號檢定 p = {main["p (sign)"]:.3f}（{sig_word(main["p (sign)"])}）；Wilcoxon p = {main["p (Wilcoxon)"]:.3f}（{sig_word(main["p (Wilcoxon)"])}）。', '',
     '【其他窗口與模型】平均 CAR（%）與 t 檢定 p 值：']
for (mname, w), r in robust.iterrows():
    L.append(f'  {mname:16s} {w:13s} N={int(r["N"]):3d}  平均 {r["Mean"]:+.2f}  p(t)={r["p (t)"]:.3f}  p(BMP)={r["p (BMP)"]:.3f}  p(Wilcoxon)={r["p (Wilcoxon)"]:.3f}')
L += ['', f'【分組比較】{MAIN_WINDOW} 的平均（%）：']
for name, r in subgroup.iterrows():
    L.append(f'  {name}：是 {r["Mean yes"]:+.2f}（N={int(r["N yes"])}）、否 {r["Mean no"]:+.2f}（N={int(r["N no"])}），差距 {r["Diff"]:+.2f}，Welch p = {r["p (Welch t)"]:.3f}，Mann-Whitney p = {r["p (Mann-Whitney)"]:.3f}')
L += ['', f'【迴歸主規格】N = {int(m1.nobs)}，Adj. R² = {m1.rsquared_adj:.3f}，F 檢定 p = {float(m1.f_pvalue):.3f}。其他條件相同時：']
for v in X_MAIN:
    b, p, lo, hi = m1.params[v], m1.pvalues[v], ci.loc[v, 0], ci.loc[v, 1]
    s = sign_summary.loc[v]
    how = f'變成 2 倍時 CAR {b * np.log(2):+.2f} 個百分點' if v.startswith('log') else f'CAR {b:+.2f} 個百分點'
    L.append(f'  {NICE[v]}：{how}（係數 {b:+.3f}，95% 信賴區間 [{lo:+.2f}, {hi:+.2f}]，p = {p:.3f}，{sig_word(p)}）；'
             f'{int(s["出現在幾個版本"])} 個版本中 {int(s["係數為正"])} 正 {int(s["係數為負"])} 負，p<0.10 的有 {int(s["p<0.10 的版本數"])} 個')
L += ['', '【不同樣本下的平均 CAR】'] + [f'  {k}：N={int(r["N"])}，平均 {r["Mean"]:+.2f}%，p(t) = {r["p (t)"]:.3f}，p(BMP) = {r["p (BMP)"]:.3f}' for k, r in rob_sample.iterrows()]
summary = '\n'.join(L)
print(summary)
with open(f'{OUT_DIR}/summary_facts.txt', 'w', encoding='utf-8') as f: f.write(summary)